In [1]:
#importing libraries
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

from src.data_loader import load_raw_data, PROCESSED_DIR

In [2]:
#loading raw data
df = load_raw_data()
df.set_index('encounter_id', inplace=True)
df.shape

(101766, 49)

In [3]:
#removing expired and hospice patients
exp_hsp_pat = [11, 13, 14, 19, 20]
df = df[~df['discharge_disposition_id'].isin(exp_hsp_pat)]
df.shape

(99343, 49)

In [4]:
#dropping weight as it has 96218 rows as blank
df.drop('weight', axis=1, inplace=True)
df.shape

(99343, 48)

In [5]:
#filling missing values with unknown
cols = ["race", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3"]
df[cols] = df[cols].fillna("Unknown")
df.isna().sum().sum()

np.int64(0)

In [6]:
#examide and citoglipton are "No" for all 99,343 patients, so we are dropping them as it constant for all patients
unq_values = df.nunique()
const_columns = unq_values[unq_values == 1].index.tolist()
print(const_columns)
df = df.drop(columns=const_columns)
df.shape

['examide', 'citoglipton']


(99343, 46)

In [7]:
#converting age map into numeric
age_map = {"[0-10)":1,"[10-20)":2,"[20-30)":3,"[30-40)":4,"[40-50)":5,"[50-60)":6,"[60-70)":7,"[70-80)":8,"[80-90)":9,"[90-100)":10}
df['age'] = df['age'].map(age_map)
df['age'].isna().sum()

np.int64(0)

In [8]:
#creating the target column
Y = (df['readmitted'] == '<30').astype(int)
df.drop('readmitted', axis=1, inplace=True)
X = df
print(Y.value_counts())
X.shape

readmitted
0    88029
1    11314
Name: count, dtype: int64


(99343, 45)

In [9]:
#dropping unknown gender
X = X[X["gender"] != "Unknown/Invalid"].copy()
Y = Y.loc[X.index]
print(X["gender"].value_counts())
X.shape, Y.shape

gender
Female    53454
Male      45886
Name: count, dtype: int64


((99340, 45), (99340,))

In [10]:
#converting categorical columns to binary columns
X["gender"] = X["gender"].map({"Female": 0, "Male": 1})
X["change"] = X["change"].map({"No": 0, "Ch": 1})
X["diabetesMed"] = X["diabetesMed"].map({"No": 0, "Yes": 1})
print(X[["gender", "change", "diabetesMed"]].isna().sum())

gender         0
change         0
diabetesMed    0
dtype: int64


In [11]:
#final check before split
print(X.shape)
print("missing values:", X.isna().sum().sum())

(99340, 45)
missing values: 0


In [12]:
#splitting 70/30 by patient
groups = X["patient_nbr"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, Y, groups))

In [13]:
#creating train and test sets
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
group_train = groups.iloc[train_idx]

In [14]:
#checking shapes, overlap and positive rate
print(X_train.shape, X_test.shape)
print("patients in both:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print(Y_train.mean().round(4), Y_test.mean().round(4))

(69613, 45) (29727, 45)
patients in both: 0
0.1137 0.1143


In [15]:
#Drop the patient id from the features
X_train = X_train.drop(columns="patient_nbr")
X_test = X_test.drop(columns="patient_nbr")
X_train.shape, X_test.shape

((69613, 44), (29727, 44))

In [16]:
#Save to Parquet
X_train.to_parquet(PROCESSED_DIR / "X_train.parquet")
X_test.to_parquet(PROCESSED_DIR / "X_test.parquet")
Y_train.to_frame("readmitted_30").to_parquet(PROCESSED_DIR / "y_train.parquet")
Y_test.to_frame("readmitted_30").to_parquet(PROCESSED_DIR / "y_test.parquet")
group_train.to_frame("patient_nbr").to_parquet(PROCESSED_DIR / "groups_train.parquet")

In [17]:
#reading back to check the files
check = pd.read_parquet(PROCESSED_DIR / "X_train.parquet")
print(check.shape, check.isna().sum().sum())
print(check[["gender", "change", "diabetesMed"]].nunique())

(69613, 44) 0
gender         2
change         2
diabetesMed    2
dtype: int64
